In [ ]:
import pandas as pd
import numpy as np

# Preprocessing

In this file we do the preprocessing of the data file `Xml_20261116_123725_1.xml`, which we obtained from a Government of Spain database. As this .xml file has the location, cause, and time stored differently, we are going to preprocess them independently, with the objective of merging them at the end.

## Location

In [ ]:
location_columns = ["numeroparte", "idcomunidad", "idprovincia",
                    "idmunicipio", "latitud", "longitud"]

locations = pd.read_xml("dades/Xml_20261116_123725_1.xml", xpath="./Pif/pif_localizacion")[location_columns]
locations["codi_municipi"] = (
    locations["idprovincia"].astype("Int64").astype("string").str.zfill(2)
    + locations["idmunicipio"].astype("Int64").astype("string").str.zfill(3)
)
locations["codi_municipi"] = locations["codi_municipi"].mask(locations["idmunicipio"]==0)
locations = locations.drop(["idcomunidad", "idprovincia", "idmunicipio"], axis=1)
locations["coords_imputades"] = (locations["latitud"].isna() & locations["codi_municipi"].notna())
locations.info()
locations.loc[locations["codi_municipi"].notna()].head()

In [ ]:
provincies = ["08", "17", "25", "43"]
columnes = ["COD_INE", "COD_PROV",
            "NOMBRE_ACTUAL",
            "PROVINCIA",
            "LONGITUD_ETRS89_REGCAN95",
            "LATITUD_ETRS89_REGCAN95"]

path = "dades/MUNICIPIOS.csv"

municipis_info = pd.read_csv(
    path,
    encoding="cp1252",
    sep=";",
    dtype="string",
)

municipis_info = municipis_info.loc[municipis_info["COD_PROV"].isin(provincies), columnes]
municipis_info["codi_municipi"] = municipis_info["COD_INE"].str[:5]
municipis_info = municipis_info.drop(["COD_INE", "COD_PROV"], axis=1)
municipis_info.info()
municipis_info.head()

In [ ]:
comarques = pd.read_csv(
    "dades/corrMpisComsAtps.csv",
    sep=";",
    skiprows=3,
    usecols=["Codi municipi", "Nom comarca", "Nom municipi"],
    dtype={"Codi municipi": "string"},
)#[["Codi municipi", "Nom comarca"]].copy()

comarques["codi_municipi"] = (
    comarques["Codi municipi"]
    .astype("string").str.strip()
    .str[:5]
    .str.zfill(5)
)
comarques = comarques[["codi_municipi", "Nom comarca"]]
comarques.info()
comarques.head()


In [ ]:
merged1 = locations.merge(
    municipis_info, 
    how="left", 
    on="codi_municipi")
merged1["latitud"] = merged1["latitud"].fillna(merged1["LATITUD_ETRS89_REGCAN95"])
merged1["longitud"] = merged1["longitud"].fillna(merged1["LONGITUD_ETRS89_REGCAN95"])
merged1 = merged1.drop(["LONGITUD_ETRS89_REGCAN95", "LATITUD_ETRS89_REGCAN95"], axis=1)
df_loc = merged1.merge(
    comarques,
    how="left",
    on="codi_municipi"
)
df_loc = df_loc.rename(columns={
    "NOMBRE_ACTUAL": "municipi",
    "PROVINCIA": "provincia",
    "Nom comarca": "comarca" 
})
df_loc = df_loc.iloc[:, [0,3,5,7,6,1,2,4]]
df_loc.loc[merged1["codi_municipi"].notna()].head()


## Causes

In [ ]:
causes = pd.read_xml("dades/Xml_20261116_123725_1.xml", xpath="./Pif/pif_causa")
causes.info()
causes.head()

In [ ]:
columnes_causes = ["numeroparte", "idcausa", "causa"]

causes_egif_ca = pd.read_csv("dades/causes_egif_ca.csv")
df_causes = causes.merge(
    causes_egif_ca,
    on="idcausa",
    how="left",
    validate="many_to_one",
)
df_causes["causa"] = df_causes["causa"].mask(df_causes["causa"] == "Desconeguda")
df_causes = df_causes[columnes_causes]
df_causes.head()


## Temps

In [ ]:
temps_columnes = ["numeroparte", "deteccion"]
df_temps = pd.read_xml("dades/Xml_20261116_123725_1.xml", xpath="./Pif/pif_tiempos")[temps_columnes]
df_temps.head()

In [ ]:
df_temps["deteccion"] = df_temps["deteccion"].astype("string").str.strip().str[:10]
df_temps["deteccion"] = pd.to_datetime(df_temps["deteccion"], errors="coerce", format="%Y-%m-%d")
df_temps.head()

## Merge and export csv file

In [ ]:
merged1 = df_loc.merge(
    df_causes,
    how="inner",
    on="numeroparte")
df = merged1.merge(
    df_temps,
    how="inner",
    on="numeroparte"
)

df.to_csv("Clean data/CleanGovEsp.csv")